In [1]:
import docs

In [2]:
def read_github_data(repo_owner, repo_name):
    allowed_extensions = {
        "md", "py", "ipynb", "yaml", "yml", "Makefile", "Dockerfile", "toml",# "json"
    }

    reader = docs.GithubRepositoryDataReader(
        repo_owner,
        repo_name,
        allowed_extensions=allowed_extensions,
    )
    
    return reader.read()

repo_owner = 'alexeygrigorev'
repo_name = 'data-engineering-rag'

files = read_github_data(repo_owner, repo_name)

In [3]:
file_index = {}

for f in files:
    file_index[f.filename] = f.content

In [4]:
import nbformat
from nbconvert import MarkdownExporter, PythonExporter
from nbconvert.preprocessors import ClearOutputPreprocessor

class NotebookMarkdownFormatter:
    """Converts Jupyter notebook content to markdown format."""

    def __init__(self):
        self.exporter = PythonExporter()
        self.exporter.register_preprocessor(ClearOutputPreprocessor(), enabled=True)

    def format(self, raw_notebook: str) -> str:
        nb_parsed = nbformat.reads(
            raw_notebook,
            as_version=nbformat.NO_CONVERT,
        )
        md_body, _ = self.exporter.from_notebook_node(nb_parsed)
        return md_body

In [5]:
notebook_processor = NotebookMarkdownFormatter()

In [6]:
file_index = {}

for f in files:
    content = f.content
    if f.filename.endswith('.ipynb'):
        content = notebook_processor.format(content)
    file_index[f.filename] = content

In [7]:
import os

class ProjectEvaluationTools:
    def __init__(self, file_index: dict[str, str]):
        self.file_index = file_index

    def read(self, filename):
        """Return the contents of a file."""
        return self.file_index.get(filename)

    def grep(self, patterns: list[str]):
        """
        Search for all patterns in all files.
        Returns: { filename: [matching lines] }
        """
        results = {}

        for filename, content in self.file_index.items():
            lines = content.splitlines()
            matches = []
            for line in lines:
                if any(p in line for p in patterns):
                    matches.append(line)
            if matches:
                results[filename] = matches

        return results

    def tree(self, wd: str = '.'):
        """
        Return a sorted list of files under the given directory.
        """
        normalized = wd.rstrip('/') + '/'
        if wd in ('', '.', './'):
            normalized = ''  # root
    
        return sorted(
            f for f in self.file_index.keys()
            if f.startswith(normalized)
        )



In [8]:
eval_tools = ProjectEvaluationTools(file_index)

In [9]:
eval_tools.tree()

['README.md',
 'agents.ipynb',
 'data-processing.ipynb',
 'debug.ipynb',
 'evals/analysis.ipynb',
 'pyproject.toml',
 'zc_agent/__init__.py',
 'zc_agent/eval/async_paralell.py',
 'zc_agent/eval/calculate_metrics.py',
 'zc_agent/eval/generate_questions.py',
 'zc_agent/eval/run_agent.py',
 'zc_agent/llm.py',
 'zc_agent/load_data.py',
 'zc_agent/logs.py',
 'zc_agent/main.py',
 'zc_agent/prepare_data.py',
 'zc_agent/prompts/__init__.py',
 'zc_agent/prompts/code_doc.md',
 'zc_agent/prompts/eval_checklist.md',
 'zc_agent/prompts/eval_question_generator.md',
 'zc_agent/prompts/eval_user.md',
 'zc_agent/prompts/notebook_edit.md',
 'zc_agent/prompts/search_agent.md',
 'zc_agent/search_agent.py',
 'zc_agent/search_tools.py']

In [10]:
from pydantic import BaseModel
from typing import Literal, Optional

class EvaluationItem(BaseModel):
    points: int
    description: str

class EvaluationCriterion(BaseModel):
    name: str
    kind: Literal["single", "checklist"]
    items: list[EvaluationItem]
    comment: Optional[str] = None

class EvaluationCriteria(BaseModel):
    criteria: list[EvaluationCriterion]

In [40]:
import yaml

In [41]:
with open('criteria.yaml', 'r') as f_in:
    criteria_raw = yaml.load(f_in, yaml.SafeLoader)

In [42]:
criteria = EvaluationCriteria.model_validate(criteria_raw)

In [43]:
criteria.criteria[0]

EvaluationCriterion(name='Dataset', kind='single', items=[EvaluationItem(points=0, description='The project uses the FAQ dataset from https://github.com/DataTalksClub/faq'), EvaluationItem(points=2, description='The project uses an original dataset')], comment="A dataset is original if it's not the FAQ dataset from DataTalks.Club. Any other dataset is considered original, also including the course repositories from DataTalks.Club like machine learning zoomcamp or MLOps zoomcamp. \n")

## Agent

In [16]:
from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat.interface import IPythonChatInterface, ChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

chat_interface = IPythonChatInterface()

In [38]:
callback = DisplayingRunnerCallback(chat_interface)

In [17]:
evaluator_tools = Tools()
raw_tools = ProjectEvaluationTools(file_index)
evaluator_tools.add_tools(raw_tools)

In [19]:
class EvaluationResult(BaseModel):
    points: int
    explanation: str

In [21]:
instructions = """
You are an expert software engineer and technical evaluator specializing in assessing GitHub
repositories against specific quality criteria.

Your mission is to thoroughly evaluate a GitHub repository and provide accurate scoring/assessment
based on the provided criteria.

Your Task:

1. Start by listing files to understand repository structure
2. Read relevant files completely based on the criteria
3. Search content across files ONLY when absolutely necessary
4. Gather concrete evidence and make your assessment
5. Provide detailed reasoning with specific examples

Tool Usage Guidelines:

- tree(): Use to see all files when you need the complete file list. It also returns the size of
  each file in characters, so avoid reading large files
- read(): Use to read specific files (README, config files, code files)
- grep(patterns): Use sparingly when searching patterns. Can accept a list of patterns to search multiple terms in ONE request


For documentation criteria: 
- Read README.md directly with read("README.md")
- Check for visuals by looking at the file list from tree()

For technical criteria:
- Use tree() ONCE to understand project structure
- Read specific files based on what you see in the file list
- Only search across files when you must find patterns in multiple locations

Investigation Guidelines:
- One tree() call shows you everything - use it wisely
- Use grep() sparingly - if you called it 3 times already, it's time to make a decision
- One file read is better than multiple file searches
- Don't search for files when you already have the complete file list
- Be evidence-based: Cite specific files, code snippets, or observations
- Don't over-investigate - sometimes the answer is obvious from the structure
- Finish after 5 interations
- Don't invoke the same tool with the same arguments more than once
- Identify the 5 most important files to read and read only them. Don't do more than 10 reads
""".strip()

In [48]:
runner = OpenAIResponsesRunner(
    tools=evaluator_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model='gpt-5-mini'),
)

In [24]:
from tqdm.auto import tqdm

In [50]:
SCORED_CRITERIA_TEMPLATE = """
Evaluate this repository against criterion "{criteria_name}"

Scoring levels:
{score_levels}

Your task: Investigate the repository and assign a score from 0 to {max_score}
points based on the levels above. Provide your score with detailed reasoning and evidence.
""".strip()

In [51]:
score_levels = '\n'.join([f'- {i.description} ({i.points} points)' for i in c.items])

if c.comment:
    score_levels = score_levels + '\n\nImprotant:\n' + c.comment

prompt = SCORED_CRITERIA_TEMPLATE.format(
    criteria_name=c.name,
    score_levels=score_levels,
    max_score=max([s.points for s in c.items]),
    comment=c.comment,
)
print(prompt)


Evaluate this repository against criterion "Dataset"

Scoring levels:
- The project uses the FAQ dataset from https://github.com/DataTalksClub/faq (0 points)
- The project uses an original dataset (2 points)

Improtant:
A dataset is original if it's not the FAQ dataset from DataTalks.Club. Any other dataset is considered original, also including the course repositories from DataTalks.Club like machine learning zoomcamp or MLOps zoomcamp. 


Your task: Investigate the repository and assign a score from 0 to 2
points based on the levels above. Provide your score with detailed reasoning and evidence.


In [ ]:
eval_result = runner.loop(
    prompt=prompt,
    output_format=EvaluationResult,
    callback=callback
)

In [56]:

def evaluate_single(c: EvaluationCriteria) -> EvaluationResult:
    score_levels = '\n'.join([f'- {i.description} ({i.points} points)' for i in c.items])

    if c.comment:
        score_levels = score_levels + '\n\nImprotant:\n' + c.comment
    
    prompt = SCORED_CRITERIA_TEMPLATE.format(
        criteria_name=c.name,
        score_levels=score_levels,
        max_score=max([s.points for s in c.items]),
        comment=c.comment,
    )
    print(prompt)
    eval_result = runner.loop(
        prompt=prompt,
        callback=callback,
        output_format=EvaluationResult
    )
    print(eval_result.cost)
    return eval_result.last_message

In [69]:
def evaluate_checklist(checklist):
    results = []
    
    for item in checklist.items:
        checklist_criterion = EvaluationCriterion(
            name=f'checklist item for "{item.description}"',
            kind='single',
            items=[
                EvaluationItem(points=0, description='checklist item not satified'),
                item
            ]
        )

        result = evaluate_single(checklist_criterion)
        results.append(result)

    return results

In [73]:
evaluate_checklist(criteria.criteria[-1])

Evaluate this repository against criterion "checklist item for "README has clear project goal and purpose""

Scoring levels:
- checklist item not satified (0 points)
- README has clear project goal and purpose (1 points)

Your task: Investigate the repository and assign a score from 0 to 1
points based on the levels above. Provide your score with detailed reasoning and evidence.
-> Response received


-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00079675'), output_cost=Decimal('0.000848'), total_cost=Decimal('0.00164475'))
Evaluate this repository against criterion "checklist item for "README includes setup/installation instructions""

Scoring levels:
- checklist item not satified (0 points)
- README includes setup/installation instructions (1 points)

Your task: Investigate the repository and assign a score from 0 to 1
points based on the levels above. Provide your score with detailed reasoning and evidence.
-> Response received


-> Response received


...

-> Response received


-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00434575'), output_cost=Decimal('0.001476'), total_cost=Decimal('0.00582175'))


[EvaluationResult(points=1, explanation='Score: 1 — README has a clear project goal and purpose.\n\nEvidence:\n- The README title and opening sentences state the intent: "This repository contains an AI Agents designed to work with data from the Data Engineering Zoomcamp repository." (README.md)\n- The Project Overview explicitly explains the goal: "This project demonstrates the creation of an AI Agent that can process and analyze content from GitHub repositories, with a focus on educational materials and code from the Data Engineering Zoomcamp course." (README.md)\n- Key Features and Supported File Types sections further clarify the purpose and capabilities (e.g., multi-format data pipeline, code-aware processing, RAG-ready data preparation), making the project goal and intended use clear.\n\nBased on these explicit statements and the documented features, the README satisfies the criterion for having a clear project goal and purpose.'),
 EvaluationResult(points=1, explanation='I inspec

In [74]:
results = [] 

for c in tqdm(criteria.criteria):
    if c.kind == 'single':
        result = evaluate_single(c)
        results.append(result)
    elif c.kind == 'checklist':
        checklist_results = evaluate_checklist(c)
        # bundle it up back to one evaluation result
        results.extend(checklist_results)

  0%|          | 0/8 [00:00<?, ?it/s]

Evaluate this repository against criterion "Dataset"

Scoring levels:
- The project uses the FAQ dataset from https://github.com/DataTalksClub/faq (0 points)
- The project uses an original dataset (2 points)

Improtant:
A dataset is original if it's not the FAQ dataset from DataTalks.Club. Any other dataset is considered original, also including the course repositories from DataTalks.Club like machine learning zoomcamp or MLOps zoomcamp. 


Your task: Investigate the repository and assign a score from 0 to 2
points based on the levels above. Provide your score with detailed reasoning and evidence.
-> Response received


-> Response received


-> Response received


-> Response received


...

CostInfo(input_cost=Decimal('0.00078775'), output_cost=Decimal('0.000748'), total_cost=Decimal('0.00153575'))
Evaluate this repository against criterion "checklist item for "Project includes visuals (demo video, screenshots, or GIFs)""

Scoring levels:
- checklist item not satified (0 points)
- Project includes visuals (demo video, screenshots, or GIFs) (1 points)

Your task: Investigate the repository and assign a score from 0 to 1
points based on the levels above. Provide your score with detailed reasoning and evidence.
-> Response received


-> Response received


-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00444525'), output_cost=Decimal('0.00108'), total_cost=Decimal('0.00552525'))


In [75]:
total_points = 0

for r in results:
    print(r)
    total_points = total_points + r.points

points=2 explanation='The repository uses the Data Engineering Zoomcamp course repository (DataTalksClub/data-engineering-zoomcamp), not the FAQ dataset. Evidence:\n\n- README.md explicitly states: "This repository contains an AI Agents designed to work with data from the Data Engineering Zoomcamp (https://github.com/DataTalksClub/data-engineering-zoomcamp) repository."\n- zc_agent/prepare_data.py downloads and processes the repo in run():\n  de_zoomcamp_data = read_repo_data(\n      "DataTalksClub",\n      "data-engineering-zoomcamp"\n  )\n  and saves to PROCESSED_DATA_PATH = "data/de-zoomcamp-processed.json".\n\nPer the scoring rules, any dataset other than the DataTalks.Club FAQ dataset (including DataTalks.Club course repos like the Zoomcamp) is considered original, so the project receives 2 points.'
points=2 explanation='Score: 2 — Well-structured data pipeline with ingestion, processing, and indexing steps.\n\nEvidence and reasoning:\n\n1) Ingestion: zc_agent/prepare_data.py impl

In [76]:
total_points

15